In [1]:
import requests
import pandas as pd
from datetime import datetime
from calendar import monthrange
import os
import time

In [2]:
def get_weather_data(start_date, end_date, api_key):
    """
    Lấy dữ liệu chất lượng không khí từ Weatherbit API.
    
    Parameters:
    -----------
    start_date : str
        Ngày bắt đầu (định dạng: 'YYYY-MM-DD')
    end_date : str
        Ngày kết thúc (định dạng: 'YYYY-MM-DD')
    api_key : str
        API key của Weatherbit
        
    Returns:
    --------
    df : pandas DataFrame hoặc None
        DataFrame chứa dữ liệu hoặc None nếu có lỗi
    """
    url = "https://api.weatherbit.io/v2.0/history/airquality"
    params = {
        'city': 'Hanoi',
        'start_date': start_date,
        'end_date': end_date,
        'tz': 'local',
        'key': api_key
    }

    try:
        # Gọi API với timeout và xử lý SSL
        response = requests.get(url, params=params, verify=False, timeout=30)
        response.raise_for_status()  # Raise exception cho HTTP errors
        
        data = response.json()
        records = data.get('data', [])
        
        if records:
            df = pd.DataFrame(records)
            print(f"Lấy được {len(df)} bản ghi từ {start_date} đến {end_date}")
            return df
        else:
            print(f"⚠️  Không có dữ liệu cho khoảng {start_date} đến {end_date}")
            return None
            
    except requests.exceptions.Timeout:
        print(f"Timeout khi lấy dữ liệu từ {start_date} đến {end_date}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Lỗi khi lấy dữ liệu từ {start_date} đến {end_date}: {e}")
        return None
    except Exception as e:
        print(f"Lỗi không xác định: {e}")
        return None

In [7]:
def save_data_to_csv(data, file_name):
    """
    Lưu dữ liệu vào file CSV.
    
    Parameters:
    -----------
    data : pandas DataFrame
        DataFrame chứa dữ liệu cần lưu
    file_name : str
        Tên file CSV
    """
    if data is not None and not data.empty:
        # Kiểm tra file đã tồn tại chưa
        file_exists = os.path.isfile(file_name)
        
        # Lưu dữ liệu (append nếu file đã tồn tại)
        data.to_csv(
            file_name, 
            index=False, 
            mode='a' if file_exists else 'w',
            header=not file_exists
        )
        
        print(f"Đã lưu {len(data)} bản ghi vào {file_name}")
        
        # Hiển thị tổng số bản ghi trong file
        if file_exists:
            total_records = len(pd.read_csv(file_name))
            print(f"Tổng số bản ghi trong file: {total_records}")
    else:
        print("Không có dữ liệu hợp lệ để lưu")

In [3]:
def get_months_between_dates(start_date, end_date):
    """
    Lấy danh sách các khoảng thời gian theo tháng giữa hai ngày.
    
    Parameters:
    -----------
    start_date : datetime
        Ngày bắt đầu
    end_date : datetime
        Ngày kết thúc
        
    Returns:
    --------
    months : list of tuples
        Danh sách các tuple (start_date, end_date) cho mỗi tháng
    """
    months = []
    start_year, start_month = start_date.year, start_date.month
    end_year, end_month = end_date.year, end_date.month

    while start_year < end_year or (start_year == end_year and start_month <= end_month):
        # Tạo ngày bắt đầu tháng
        month_start = datetime(start_year, start_month, 1)
        
        # Lấy ngày cuối tháng
        _, last_day = monthrange(start_year, start_month)
        month_end = datetime(start_year, start_month, last_day)
        
        # Đảm bảo không vượt quá end_date
        if month_end > end_date:
            month_end = end_date

        months.append((
            month_start.strftime('%Y-%m-%d'), 
            month_end.strftime('%Y-%m-%d')
        ))

        # Chuyển sang tháng tiếp theo
        if start_month == 12:
            start_month = 1
            start_year += 1
        else:
            start_month += 1

    return months

In [5]:
# Cấu hình
API_KEY = 'bd78df23972d4c5787e72fd978e7c5cb'
OUTPUT_FILE = 'weather_data_hanoi_test_data.csv'

# Xác định khoảng thời gian
start_date = datetime(2025, 11, 10)
end_date = datetime(2025, 11, 17)

print("="*60)
print("CẤU HÌNH LẤY DỮ LIỆU THỜI TIẾT")
print("="*60)
print(f"Từ ngày: {start_date.strftime('%Y-%m-%d')}")
print(f"Đến ngày: {end_date.strftime('%Y-%m-%d')}")
print(f"File output: {OUTPUT_FILE}")

# Lấy danh sách các tháng
months = get_months_between_dates(start_date, end_date)
print(f"\nTổng số tháng cần lấy dữ liệu: {len(months)}")
print(f"Danh sách các khoảng thời gian:")
for i, (month_start, month_end) in enumerate(months[:5], 1):
    print(f"   {i}. {month_start} → {month_end}")
if len(months) > 5:
    print(f"   ... và {len(months) - 5} tháng khác")
print("="*60)

CẤU HÌNH LẤY DỮ LIỆU THỜI TIẾT
Từ ngày: 2025-11-10
Đến ngày: 2025-11-17
File output: weather_data_hanoi_test_data.csv

Tổng số tháng cần lấy dữ liệu: 1
Danh sách các khoảng thời gian:
   1. 2025-11-01 → 2025-11-17


In [8]:
print("\n" + "="*60)
print("BẮT ĐẦU LẤY DỮ LIỆU")
print("="*60 + "\n")

# Thống kê
total_months = len(months)
success_count = 0
failed_count = 0
total_records = 0

# Lấy dữ liệu cho từng tháng
for index, (month_start, month_end) in enumerate(months, 1):
    print(f"\n[{index}/{total_months}] 📡 Đang lấy dữ liệu: {month_start} → {month_end}")
    
    # Lấy dữ liệu từ API
    data = get_weather_data(month_start, month_end, API_KEY)
    
    # Lưu dữ liệu
    if data is not None:
        save_data_to_csv(data, OUTPUT_FILE)
        success_count += 1
        total_records += len(data)
    else:
        failed_count += 1
    
    # Delay để tránh rate limiting (nếu cần)
    if index < total_months:
        time.sleep(1)  # Chờ 1 giây trước khi request tiếp theo

# Tóm tắt kết quả
print("\n" + "="*60)
print("KẾT QUẢ LẤY DỮ LIỆU")
print("="*60)
print(f"Thành công: {success_count}/{total_months} tháng")
print(f"Thất bại: {failed_count}/{total_months} tháng")
print(f"Tổng số bản ghi: {total_records}")
print(f"File đã lưu: {OUTPUT_FILE}")
print("="*60)

# Đọc và hiển thị thông tin file CSV
if os.path.isfile(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)
    print(f"\nTHÔNG TIN FILE CSV:")
    print(f"   - Tổng số dòng: {len(df)}")
    print(f"   - Tổng số cột: {len(df.columns)}")
    print(f"   - Các cột: {list(df.columns[:10])}")
    if len(df.columns) > 10:
        print(f"              ...và {len(df.columns) - 10} cột khác")
    print(f"\n5 dòng đầu tiên:")
    print(df.head())


BẮT ĐẦU LẤY DỮ LIỆU


[1/1] 📡 Đang lấy dữ liệu: 2025-11-01 → 2025-11-17


c:\Users\63200744\AppData\Local\Programs\Python\Python313\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.weatherbit.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Lấy được 385 bản ghi từ 2025-11-01 đến 2025-11-17
Đã lưu 385 bản ghi vào weather_data_hanoi_test_data.csv

KẾT QUẢ LẤY DỮ LIỆU
Thành công: 1/1 tháng
Thất bại: 0/1 tháng
Tổng số bản ghi: 385
File đã lưu: weather_data_hanoi_test_data.csv

THÔNG TIN FILE CSV:
   - Tổng số dòng: 385
   - Tổng số cột: 11
   - Các cột: ['aqi', 'co', 'datetime', 'no2', 'o3', 'pm10', 'pm25', 'so2', 'timestamp_local', 'timestamp_utc']
              ...và 1 cột khác

5 dòng đầu tiên:
   aqi      co       datetime   no2    o3   pm10    pm25   so2  \
0  211  1198.0  2025-11-16:17  60.3  50.2  135.7  101.67  12.0   
1  227  1625.3  2025-11-16:16  43.3  17.2  146.0  113.60  12.3   
2  208  2361.0  2025-11-16:15  38.3  18.5  137.0   99.00  25.3   
3  196  1210.0  2025-11-16:14  15.0   3.6  127.7   90.33  63.0   
4  191  4766.0  2025-11-16:13  70.5  36.7  116.7   86.00  35.7   

       timestamp_local        timestamp_utc          ts  
0  2025-11-17T00:00:00  2025-11-16T17:00:00  1763312400  
1  2025-11-16T23:00:00  2